In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q16/q16_rewrite/checkpoints/pre_cell_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 3 ###

# 1) Pre-compute the supplier keys to exclude via a single filter & drop_duplicates
sup_keys = (
    supplier
    .loc[
        supplier.S_COMMENT.str.contains("Customer.*Complaints"),
        "S_SUPPKEY"
    ]
    .drop_duplicates()
)

# 2–5) Filter, join, exclude and aggregate in one chained pipeline
total = (
    part
    .loc[
        (part.P_BRAND != "Brand#45")
        & ~part.P_TYPE.str.contains("^MEDIUM POLISHED")
        & part.P_SIZE.isin([49, 14, 23, 45, 19, 3, 36, 9]),
        ["P_PARTKEY", "P_BRAND", "P_TYPE", "P_SIZE"]
    ]
    .merge(
        partsupp[["PS_PARTKEY", "PS_SUPPKEY"]],
        left_on="P_PARTKEY",
        right_on="PS_PARTKEY"
    )
    .loc[lambda df: ~df.PS_SUPPKEY.isin(sup_keys)]
    .groupby(
        ["P_BRAND", "P_TYPE", "P_SIZE"],
        as_index=False,
        sort=False
    )
    .agg(SUPPLIER_CNT=("PS_SUPPKEY", "nunique"))
    .sort_values(
        by=["SUPPLIER_CNT", "P_BRAND", "P_TYPE", "P_SIZE"],
        ascending=[False, True, True, True]
    )
)
